# Two-Branch LSTM Fraud Detection — Behavior Agent (Nepal Digital Banking)

A **many-to-one** LSTM that predicts `is_fraud` for the *current* transaction from that
account's chronological history, fused with an account-level static branch.

```
 per-account txn sequence ──► LSTM ──► last hidden state ┐
                                                         ├─► concat ─► dense ─► sigmoid
 account static features ────► dense embedding ──────────┘
```

**Ground-truth data facts** this notebook is built around (already verified — not re-derived):
- `feature_table.csv` = **2,000,000** rows / **50,000** accounts (400,001 labeled + 1,599,999 unlabeled).
- Full txns-per-account: mean ~40, min 18, max 68 — only 6.9% of accounts reach 50 txns.
  **Therefore sequence length N lives in [20, 35]; N=30 default.** Pushing N→50 makes >90% padding.
- Labels: 7,338 fraud (**1.83%**) among the 400k labeled rows. The other 1.6M are unlabeled
  holdout used **only as sequence context**, never as prediction targets.
- Device timestamps: 50.2% of `device_fingerprints.json` rows have `first_seen > last_seen`.
  This was fixed **upstream** during cleaning (robust `min(first_seen,last_seen)` derivation);
  we reuse the cleaned/joined `feature_table.csv`, so device features are already correct.

**Reality check (documented, not a bug):** prior diagnostics on this exact synthetic dump show
~89.7% of fraud is statistically indistinguishable from legit traffic; every offline model
(XGBoost/LightGBM/IsoForest/LSTM) caps at **AUROC ~0.52–0.56** per-transaction. This notebook is
built *correctly* (leakage-controlled, time-split, masked) but will hit that same data ceiling —
the value is a faithful, reusable pipeline, not a headline score.

**Leakage rules enforced below (non-negotiable):** never use `fraud_type`/`fraud_confidence`
(post-hoc), never use `rule_*` or model-verdict outputs as inputs (benchmark only), never use the
late customer-profile snapshot fields (`risk_tier`, `churn_risk_score`, `avg_monthly_*`,
`is_dormant`), exclude `is_fraud_seed`, and **time-ordered split** with all scalers/encoders fit
on TRAIN only.

## 0 · Configuration — all paths & hyperparameters (single source of truth)

In [1]:
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, f1_score
import joblib
warnings.filterwarnings("ignore")

# ---- Paths: resolve the backend/ root by walking up until we see datasets_processed/ ----
def _find_backend():
    p = os.path.abspath(os.getcwd())
    for _ in range(8):
        if os.path.isdir(os.path.join(p, "datasets_processed")):
            return p
        p = os.path.dirname(p)
    raise FileNotFoundError("Could not locate a 'datasets_processed' directory above cwd")

BACKEND      = _find_backend()
DATA_DIR     = os.environ.get("DATA_DIR", os.path.join(BACKEND, "datasets_processed"))
RAW_DIR      = os.environ.get("RAW_DIR",  os.path.join(BACKEND, "datasets"))
PROCESSED_DIR= os.path.join(DATA_DIR, "lstm")          # LSTM index artifacts
MODELS_DIR   = os.path.join(BACKEND, "models", "lstm") # model + scalers + manifest
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

FULL_FEATURE_TABLE = os.path.join(DATA_DIR, "feature_table.csv")     # all 2M txns
CUSTOMER_PROFILES  = os.path.join(DATA_DIR, "customer_profiles_cleaned.csv")
GRAPH_NODES        = os.path.join(DATA_DIR, "account_graph_nodes.csv")
DEVICE_JSON        = os.path.join(RAW_DIR,  "device_fingerprints.json")
RULE_BASELINE      = os.path.join(RAW_DIR,  "rule_engine_baseline_predictions.csv")  # benchmark only
MODEL_VERDICTS     = os.path.join(RAW_DIR,  "model_verdicts_sample.json")            # benchmark only

# ---- Ground-truth assertions (this file has silently truncated on upload before) ----
EXPECTED_FULL_ROWS   = 2_000_000
EXPECTED_ACCOUNTS    = 50_000
EXPECTED_LABELED     = 400_001
EXPECTED_FRAUD       = 7_338

# ---- Sequence length: MUST be in [20,35]; 30 is default and already near the ceiling ----
SEQ_LEN = 30
assert 20 <= SEQ_LEN <= 35, "N must stay in [20,35] per the txns-per-account distribution"

# ---- Leakage-control column sets ----
KEY_COLS            = ["txn_id", "timestamp", "account_id", "month", "is_fraud"]
LEAKAGE_POSTHOC     = ["fraud_type", "fraud_confidence"]                       # rule 1
LEAKAGE_OTHER_SYS   = ["rule_baseline_decision", "rule_triggered", "rule_confidence"]  # rule 2
LEAKAGE_SNAPSHOT    = ["cust_avg_monthly_txn_count", "cust_avg_monthly_txn_value_npr",  # rule 3
                       "cust_churn_risk_score", "cust_is_dormant",
                       "cust_risk_tier_HIGH", "cust_risk_tier_LOW",
                       "cust_risk_tier_MEDIUM", "cust_risk_tier_WATCHLIST"]
DROP_ID_STRING      = ["counterparty_id", "device_id", "ip_address", "terminal_id", "session_id",
                       "merchant_category_code", "geo_ip_country", "dev_locale",
                       "otp_trigger_reason", "otp_final_decision"]
STATIC_IN_TABLE     = ["cust_kyc_tier_BASIC", "cust_kyc_tier_ENHANCED", "cust_kyc_tier_STANDARD"]
RAW_AMOUNT, LOG_AMOUNT = "amount_npr", "log_amount_npr"
SEQ_EXCLUDE = set(KEY_COLS + LEAKAGE_POSTHOC + LEAKAGE_OTHER_SYS + LEAKAGE_SNAPSHOT
                  + DROP_ID_STRING + STATIC_IN_TABLE + [RAW_AMOUNT])

# ---- Static branch (SAFE fields only, rule 3) ----
STATIC_CAT_ONEHOT = ["age_group", "province", "occupation_category", "monthly_income_band_npr", "kyc_tier"]
STATIC_CAT_FREQ   = ["district"]                     # ~77 districts -> frequency-encode on TRAIN
STATIC_NUM_PROFILE= ["has_linked_esewa", "has_linked_khalti", "num_beneficiaries_registered", "account_age_days"]
STATIC_NUM_GRAPH  = ["degree_in", "degree_out", "total_received_npr", "total_sent_npr"]  # is_fraud_seed EXCLUDED

# ---- Time-ordered split by calendar month of the ENDING txn (never random) ----
TRAIN_MONTHS = list(range(1, 9))   # 1-8
VAL_MONTHS   = [9, 10]
TEST_MONTHS  = [11, 12]

# ---- Hyperparameters ----
LSTM_HIDDEN, LSTM_LAYERS = 64, 1
STATIC_EMB, FUSION_HIDDEN, DROPOUT = 32, 64, 0.2
BATCH_SIZE, EPOCHS, LR, WEIGHT_DECAY = 512, 4, 1e-3, 1e-5
POS_WEIGHT = 53.5                  # (1-p)/p for p=1.83%
TRAIN_NEG_PER_POS = 10             # CPU-frugal: undersample legit windows in TRAIN only
TARGET_PRECISION = 0.20
SEED = 42

np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(max(1, (os.cpu_count() or 2)))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("backend:", BACKEND)
print("device :", DEVICE, "| torch", torch.__version__, "| threads", torch.get_num_threads())
print("N (seq len):", SEQ_LEN, "| split  train", TRAIN_MONTHS, "val", VAL_MONTHS, "test", TEST_MONTHS)


backend: C:\Developer\Agentic-Fraud-Detection-System-\backend
device : cpu | torch 2.12.1+cpu | threads 8
N (seq len): 30 | split  train [1, 2, 3, 4, 5, 6, 7, 8] val [9, 10] test [11, 12]


## 1 · Data audit — verify ground-truth facts before any modeling
Prints a checkpoint for every fact. **Stops** (raises) if the row count is short or the
fraud rate is materially off, rather than silently proceeding.

In [2]:
t0 = time.time()
print("=== STAGE 1: DATA AUDIT ===")

# Row count / accounts / labels / fraud rate — stream just the tiny columns.
n_rows = n_fraud = n_labeled = 0
acct_counts = {}
month_counts = {}
for ch in pd.read_csv(FULL_FEATURE_TABLE, usecols=["account_id", "is_fraud", "month"], chunksize=500_000):
    n_rows += len(ch)
    lab = ch["is_fraud"].notna()
    n_labeled += int(lab.sum())
    n_fraud += int(np.nansum(ch["is_fraud"].values))
    for a, c in ch["account_id"].value_counts().items():
        acct_counts[a] = acct_counts.get(a, 0) + int(c)
    for m, c in ch.loc[lab, "month"].value_counts().items():
        month_counts[m] = month_counts.get(m, 0) + int(c)

tpa = np.array(list(acct_counts.values()))
fraud_rate = n_fraud / max(n_labeled, 1)
print(f"[audit] full rows ............ {n_rows:,}  (expect {EXPECTED_FULL_ROWS:,})")
print(f"[audit] accounts ............. {len(acct_counts):,}  (expect {EXPECTED_ACCOUNTS:,})")
print(f"[audit] labeled rows ......... {n_labeled:,}  (expect {EXPECTED_LABELED:,})")
print(f"[audit] fraud rows ........... {n_fraud:,}  (expect {EXPECTED_FRAUD:,})  rate {fraud_rate:.4%}")
print(f"[audit] txns/account ......... mean {tpa.mean():.1f}  min {tpa.min()}  max {tpa.max()}  std {tpa.std():.1f}")
print(f"[audit] pct accts >=20 txns .. {(tpa>=20).mean():.1%}   >=30: {(tpa>=30).mean():.1%}   >=50: {(tpa>=50).mean():.1%}")
print(f"[audit] labeled month dist ... {dict(sorted(month_counts.items()))}")

# STOP conditions
assert n_rows == EXPECTED_FULL_ROWS, f"ROW COUNT SHORT: {n_rows} != {EXPECTED_FULL_ROWS} — file truncated, STOP."
assert abs(fraud_rate - 0.0183) < 0.004, f"Fraud rate {fraud_rate:.4%} off expected 1.83% — STOP."
assert tpa.max() <= 68 and tpa.min() >= 15, "txns/account distribution off expected 18..68 — STOP."

# Device reversed-timestamp fact (~50%) — demonstrates why the upstream fix was needed.
try:
    dev = pd.read_json(DEVICE_JSON)
    fs = pd.to_datetime(dev["first_seen"], errors="coerce", utc=True)
    ls = pd.to_datetime(dev["last_seen"], errors="coerce", utc=True)
    rev = (fs > ls).mean()
    print(f"[audit] device first_seen>last_seen (reversed): {rev:.1%}  (fixed upstream in cleaning)")
except Exception as e:
    print("[audit] device json check skipped:", e)

# Join coverage: static-branch sources.
prof_ids = set(pd.read_csv(CUSTOMER_PROFILES, usecols=["account_id"])["account_id"])
node_df  = pd.read_csv(GRAPH_NODES)
node_ids = set(node_df["id"])
all_accts = set(acct_counts)
print(f"[audit] customer_profiles coverage . {len(all_accts & prof_ids)/len(all_accts):.1%}")
print(f"[audit] graph_nodes coverage ....... {len(all_accts & node_ids)/len(all_accts):.1%}")
print(f"[audit] OK — proceeding. ({time.time()-t0:.1f}s)")


=== STAGE 1: DATA AUDIT ===
[audit] full rows ............ 2,000,000  (expect 2,000,000)
[audit] accounts ............. 50,000  (expect 50,000)
[audit] labeled rows ......... 400,001  (expect 400,001)
[audit] fraud rows ........... 7,338  (expect 7,338)  rate 1.8345%
[audit] txns/account ......... mean 40.0  min 18  max 68  std 6.3
[audit] pct accts >=20 txns .. 100.0%   >=30: 95.8%   >=50: 6.9%
[audit] labeled month dist ... {1: 48259, 2: 43236, 3: 47934, 4: 46470, 5: 48176, 6: 23319, 7: 24029, 8: 24106, 9: 23177, 10: 24136, 11: 23133, 12: 24026}
[audit] device first_seen>last_seen (reversed): 50.2%  (fixed upstream in cleaning)
[audit] customer_profiles coverage . 100.0%
[audit] graph_nodes coverage ....... 100.0%
[audit] OK — proceeding. (13.9s)


## 2 · Load features with leakage control
Read the full 2M-row table keeping only numeric feature columns + keys (skip high-cardinality
ID/string columns at parse time). Drop every leakage column, coerce booleans to 0/1, and replace
raw `amount_npr` with `log1p(amount)`.

In [3]:
print("=== STAGE 2: LOAD + LEAKAGE DROP ===")
t0 = time.time()
header = pd.read_csv(FULL_FEATURE_TABLE, nrows=0).columns.tolist()

# String/leakage columns we skip at parse time (saves memory); keep keys + numeric features + raw amount.
_string_or_leak = set(DROP_ID_STRING + LEAKAGE_POSTHOC + LEAKAGE_OTHER_SYS)
usecols = [c for c in header if c not in _string_or_leak]
df = pd.read_csv(FULL_FEATURE_TABLE, usecols=usecols)
print(f"[load] read {df.shape[0]:,} x {df.shape[1]} cols in {time.time()-t0:.1f}s")

# Coerce booleans -> int8; everything except keys/timestamp is a numeric feature.
for c in df.columns:
    if df[c].dtype == bool:
        df[c] = df[c].astype("int8")

# log1p(amount) as a scaled sequential feature; drop the raw amount.
df[LOG_AMOUNT] = np.log1p(df[RAW_AMOUNT].clip(lower=0).fillna(0.0))

# Final per-timestep sequential feature list = everything not excluded, deterministic order.
SEQ_FEATURES = [c for c in df.columns if c not in SEQ_EXCLUDE]
# Numeric-coerce + fill; keep as float32 to bound memory (2M x ~80).
for c in SEQ_FEATURES:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
df[SEQ_FEATURES] = df[SEQ_FEATURES].fillna(0.0)

print(f"[load] sequential features: {len(SEQ_FEATURES)}")
print(f"[load] excluded for leakage: posthoc={LEAKAGE_POSTHOC} | other_sys={LEAKAGE_OTHER_SYS}")
print(f"[load]                       snapshot={LEAKAGE_SNAPSHOT}")
print(f"[load] first 12 seq feats: {SEQ_FEATURES[:12]}")
assert RAW_AMOUNT not in SEQ_FEATURES and "fraud_type" not in SEQ_FEATURES


=== STAGE 2: LOAD + LEAKAGE DROP ===
[load] read 2,000,000 x 96 cols in 20.8s
[load] sequential features: 80
[load] excluded for leakage: posthoc=['fraud_type', 'fraud_confidence'] | other_sys=['rule_baseline_decision', 'rule_triggered', 'rule_confidence']
[load]                       snapshot=['cust_avg_monthly_txn_count', 'cust_avg_monthly_txn_value_npr', 'cust_churn_risk_score', 'cust_is_dormant', 'cust_risk_tier_HIGH', 'cust_risk_tier_LOW', 'cust_risk_tier_MEDIUM', 'cust_risk_tier_WATCHLIST']
[load] first 12 seq feats: ['response_code', 'processing_time_ms', 'is_international', 'fx_rate', 'has_device_id', 'has_terminal_id', 'has_session_id', 'has_fx_rate', 'has_notes', 'is_malformed_ip', 'tz_suspect', 'is_invalid_amount']


## 3 · Sort chronologically & index per-account row ranges
Sort by `account_id` then `timestamp` so each account's rows are contiguous and time-ordered.
Windows are sliced from these contiguous ranges — using only that account's own past.

In [4]:
print("=== STAGE 3: SORT + ACCOUNT OFFSETS ===")
t0 = time.time()
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values(["account_id", "timestamp"], kind="mergesort").reset_index(drop=True)

# Contiguous account blocks -> (start,end) offsets; codes are monotonic because we sorted by account.
acct_ids = df["account_id"].values
codes, uniques = pd.factorize(acct_ids, sort=False)   # already grouped by the sort
n_accounts = len(uniques)
# starts: first index of each code
starts = np.zeros(n_accounts, dtype=np.int64)
boundaries = np.flatnonzero(np.diff(codes) != 0) + 1
starts[1:] = boundaries
ends = np.empty(n_accounts, dtype=np.int64)
ends[:-1] = boundaries
ends[-1] = len(df)
acct_index_of_row = codes.astype(np.int32)            # per-row account index

print(f"[sort] rows {len(df):,} | accounts {n_accounts:,} | {time.time()-t0:.1f}s")
print(f"[sort] block sizes: mean {(ends-starts).mean():.1f} min {(ends-starts).min()} max {(ends-starts).max()}")
assert n_accounts == EXPECTED_ACCOUNTS


=== STAGE 3: SORT + ACCOUNT OFFSETS ===
[sort] rows 2,000,000 | accounts 50,000 | 9.1s
[sort] block sizes: mean 40.0 min 18 max 68


## 4 · Static branch — SAFE account-level features
From `customer_profiles` keep only fields that do **not** change from fraud outcomes; from
`account_graph_nodes` keep degree/flow (**exclude `is_fraud_seed`**). `account_age_days` is
computed as (account's last txn time − `customer_since`), clipped ≥ 0.

In [5]:
print("=== STAGE 4: STATIC BRANCH ===")
t0 = time.time()
# Per-account last-txn timestamp (rows are sorted, so ends-1 is the account's latest).
last_ts = df["timestamp"].values[ends - 1]
acct_order = pd.Index(uniques, name="account_id")

prof = pd.read_csv(CUSTOMER_PROFILES).drop_duplicates("account_id").set_index("account_id")
prof = prof.reindex(acct_order)
cust_since = pd.to_datetime(prof["customer_since"], errors="coerce").values
account_age_days = (pd.to_datetime(last_ts) - pd.to_datetime(cust_since)).days if False else None
# vectorised day diff, clip >= 0
account_age_days = (last_ts.astype("datetime64[ns]") - cust_since.astype("datetime64[ns]")) / np.timedelta64(1, "D")
account_age_days = np.clip(np.nan_to_num(account_age_days, nan=0.0), 0, None)

nodes = pd.read_csv(GRAPH_NODES).drop_duplicates("id").set_index("id").reindex(acct_order)

static_raw = pd.DataFrame(index=acct_order)
for c in STATIC_CAT_ONEHOT + STATIC_CAT_FREQ:
    static_raw[c] = prof[c].astype("object").fillna("UNK").values
static_raw["has_linked_esewa"] = prof["has_linked_esewa"].astype("float32").fillna(0).values
static_raw["has_linked_khalti"] = prof["has_linked_khalti"].astype("float32").fillna(0).values
static_raw["num_beneficiaries_registered"] = pd.to_numeric(prof["num_beneficiaries_registered"], errors="coerce").fillna(0).astype("float32").values
static_raw["account_age_days"] = account_age_days.astype("float32")
for c in STATIC_NUM_GRAPH:
    static_raw[c] = pd.to_numeric(nodes[c], errors="coerce").fillna(0).astype("float32").values

print(f"[static] rows {static_raw.shape[0]:,} x raw cols {static_raw.shape[1]} | {time.time()-t0:.1f}s")
print(f"[static] EXCLUDED is_fraud_seed (leakage rule 4):", "is_fraud_seed" not in static_raw.columns)
print(static_raw.head(3).to_string())


=== STAGE 4: STATIC BRANCH ===
[static] rows 50,000 x raw cols 14 | 0.4s
[static] EXCLUDED is_fraud_seed (leakage rule 4): True
            age_group province occupation_category monthly_income_band_npr  kyc_tier    district  has_linked_esewa  has_linked_khalti  num_beneficiaries_registered  account_age_days  degree_in  degree_out  total_received_npr  total_sent_npr
account_id                                                                                                                                                                                                                                      
ACC-1000159     25-34  Gandaki            SALARIED                   200K+  ENHANCED     Pokhara               0.0                1.0                          14.0       1133.610352      139.0       144.0          66853544.0      73689984.0
ACC-1000599     45-54    Koshi             STUDENT                    <15K  ENHANCED  Biratnagar               1.0                0.0                

## 5 · Time-ordered split & fit scalers/encoders on TRAIN only
Window-end rows are the labeled txns; each is assigned to train/val/test by its calendar month.
The sequential `StandardScaler` is fit on **all rows in TRAIN months** (labeled + unlabeled
context) and applied everywhere. Static encoders/scalers are fit on **train accounts only**.

In [6]:
print("=== STAGE 5: TIME SPLIT + FIT (TRAIN ONLY) ===")
t0 = time.time()
month = df["month"].values
is_end = df["is_fraud"].notna().values                 # labeled rows = valid window ends
split_of_month = {}
for m in TRAIN_MONTHS: split_of_month[m] = 0
for m in VAL_MONTHS:   split_of_month[m] = 1
for m in TEST_MONTHS:  split_of_month[m] = 2
row_split = np.vectorize(lambda m: split_of_month.get(int(m), -1))(month).astype(np.int8)

# ---- Sequential scaler: fit on TRAIN-month rows only, transform all ----
train_row_mask = (row_split == 0)
X = df[SEQ_FEATURES].values.astype(np.float32)
seq_scaler = StandardScaler().fit(X[train_row_mask])
X = seq_scaler.transform(X).astype(np.float32)         # scaled sequential matrix (2M x F_seq)
print(f"[split] seq matrix {X.shape} scaled on {train_row_mask.sum():,} train rows")

# ---- Static encoders: fit on TRAIN accounts only ----
end_rows = np.flatnonzero(is_end)
end_acct = acct_index_of_row[end_rows]
end_split = row_split[end_rows]
train_accts = np.unique(end_acct[end_split == 0])
train_acct_mask = np.zeros(n_accounts, dtype=bool); train_acct_mask[train_accts] = True

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32)
ohe.fit(static_raw.loc[train_acct_mask, STATIC_CAT_ONEHOT])
ohe_all = ohe.transform(static_raw[STATIC_CAT_ONEHOT])

# district frequency encoding from TRAIN accounts
freq = static_raw.loc[train_acct_mask, "district"].value_counts(normalize=True)
district_freq = static_raw["district"].map(freq).fillna(0.0).astype("float32").values.reshape(-1, 1)

num_cols = STATIC_NUM_PROFILE + STATIC_NUM_GRAPH
num_mat = np.hstack([static_raw[num_cols].values.astype(np.float32), district_freq])
stat_scaler = StandardScaler().fit(num_mat[train_acct_mask])
num_scaled = stat_scaler.transform(num_mat).astype(np.float32)

STATIC_MATRIX = np.hstack([ohe_all, num_scaled]).astype(np.float32)
STATIC_FEATURE_NAMES = list(ohe.get_feature_names_out(STATIC_CAT_ONEHOT)) + num_cols + ["district_freq"]
print(f"[split] static matrix {STATIC_MATRIX.shape} | train accounts {train_acct_mask.sum():,}")

# ---- Window index arrays (all labeled rows) ----
win_end   = end_rows.astype(np.int64)
win_acct  = end_acct.astype(np.int32)
win_label = df["is_fraud"].values[end_rows].astype(np.int8)
win_split = end_split.astype(np.int8)
for name, s in [("train", 0), ("val", 1), ("test", 2)]:
    m = win_split == s
    print(f"[split] {name:5s}: windows {m.sum():,}  fraud {win_label[m].sum():,}  rate {win_label[m].mean():.4%}")
print(f"[split] done {time.time()-t0:.1f}s")


=== STAGE 5: TIME SPLIT + FIT (TRAIN ONLY) ===
[split] seq matrix (2000000, 80) scaled on 1,527,701 train rows
[split] static matrix (50000, 32) | train accounts 49,876
[split] train: windows 305,529  fraud 5,553  rate 1.8175%
[split] val  : windows 47,313  fraud 902  rate 1.9065%
[split] test : windows 47,159  fraud 883  rate 1.8724%
[split] done 3.1s


## 6 · Window index + TRAIN-only negative undersampling
Class imbalance is handled two ways: weighted BCE (`pos_weight≈53.5`) **and** CPU-frugal
undersampling of legit windows in the **train** split only (keep every fraud window). Val/test
keep **every** window for honest evaluation.

In [7]:
print("=== STAGE 6: WINDOW INDEX + UNDERSAMPLE ===")
rng = np.random.RandomState(SEED)
idx_all = np.arange(len(win_end))
tr = idx_all[win_split == 0]
tr_pos = tr[win_label[tr] == 1]
tr_neg = tr[win_label[tr] == 0]
keep_neg = rng.choice(tr_neg, size=min(len(tr_neg), TRAIN_NEG_PER_POS * len(tr_pos)), replace=False)
train_idx = np.concatenate([tr_pos, keep_neg]); rng.shuffle(train_idx)
val_idx  = idx_all[win_split == 1]
test_idx = idx_all[win_split == 2]
print(f"[idx] train windows {len(train_idx):,} (pos {len(tr_pos):,} / neg {len(keep_neg):,}) "
      f"| val {len(val_idx):,} | test {len(test_idx):,}")
print(f"[idx] train pos rate {win_label[train_idx].mean():.2%} (undersampled 1:{TRAIN_NEG_PER_POS})")


=== STAGE 6: WINDOW INDEX + UNDERSAMPLE ===
[idx] train windows 61,083 (pos 5,553 / neg 55,530) | val 47,313 | test 47,159
[idx] train pos rate 9.09% (undersampled 1:10)


## 7 · Windowed `torch.Dataset` (left-pad + mask)
`__getitem__` slices up to the last `N` rows of the account ending at the window row, then
**left-pads** with zero vectors to length `N` and returns a boolean mask so padding contributes
nothing to the LSTM (via `pack_padded_sequence` in the model).

In [8]:
class WindowDataset(Dataset):
    """Lazy per-window slicing over the shared scaled matrices — no giant 3-D tensor."""
    def __init__(self, indices):
        self.indices = np.asarray(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        w = self.indices[i]
        end = win_end[w]                      # global row index of the current txn
        a = win_acct[w]
        start = max(starts[a], end - SEQ_LEN + 1)
        seq = X[start:end + 1]                # (L, F_seq), L <= SEQ_LEN, chronological
        L = seq.shape[0]
        padded = np.zeros((SEQ_LEN, X.shape[1]), dtype=np.float32)
        padded[SEQ_LEN - L:] = seq            # LEFT-pad: real steps sit at the end
        mask = np.zeros(SEQ_LEN, dtype=np.float32); mask[SEQ_LEN - L:] = 1.0
        stat = STATIC_MATRIX[a]
        y = np.float32(win_label[w])
        return (torch.from_numpy(padded), torch.tensor(L, dtype=torch.long),
                torch.from_numpy(stat), torch.tensor(y))

def make_loader(indices, shuffle):
    return DataLoader(WindowDataset(indices), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, drop_last=False)

train_loader = make_loader(train_idx, True)
val_loader   = make_loader(val_idx,  False)
test_loader  = make_loader(test_idx, False)
xb, lb, sb, yb = next(iter(train_loader))
print("[dataset] batch shapes:", tuple(xb.shape), "lengths", tuple(lb.shape),
      "static", tuple(sb.shape), "labels", tuple(yb.shape))
print("[dataset] N_SEQ_FEAT =", X.shape[1], "| N_STATIC_FEAT =", STATIC_MATRIX.shape[1])


[dataset] batch shapes: (512, 30, 80) lengths (512,) static (512, 32) labels (512,)
[dataset] N_SEQ_FEAT = 80 | N_STATIC_FEAT = 32


## 8 · Two-branch model
Sequential branch: `pack_padded_sequence` → LSTM → last real hidden state (padding ignored).
Static branch: dense embedding. Fusion: concat → dense → single logit (BCE-with-logits).

In [9]:
class TwoBranchLSTM(nn.Module):
    def __init__(self, n_seq_feat, n_static_feat, hidden=LSTM_HIDDEN, layers=LSTM_LAYERS,
                 static_emb=STATIC_EMB, fusion_hidden=FUSION_HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_seq_feat, hidden, num_layers=layers, batch_first=True)
        self.static = nn.Sequential(nn.Linear(n_static_feat, static_emb), nn.ReLU(), nn.Dropout(dropout))
        self.head = nn.Sequential(
            nn.Linear(hidden + static_emb, fusion_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1))
    def forward(self, seq, lengths, static):
        packed = nn.utils.rnn.pack_padded_sequence(seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)       # h_n[-1] = last layer's final hidden state (masking-safe)
        fused = torch.cat([h_n[-1], self.static(static)], dim=1)
        return self.head(fused).squeeze(1)    # raw logits

model = TwoBranchLSTM(X.shape[1], STATIC_MATRIX.shape[1]).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"[model] trainable params: {n_params:,}")


TwoBranchLSTM(
  (lstm): LSTM(80, 64, batch_first=True)
  (static): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=96, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
[model] trainable params: 44,705


## 9 · Train — weighted BCE, PR-AUC as the selection metric
`pos_weight≈53.5` up-weights the 1.83% fraud class. We select the epoch with the best
**validation PR-AUC** (never accuracy).

In [10]:
@torch.no_grad()
def evaluate_probs(loader):
    model.eval()
    ys, ps = [], []
    for seq, lengths, stat, y in loader:
        logits = model(seq.to(DEVICE), lengths, stat.to(DEVICE))
        ps.append(torch.sigmoid(logits).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print("=== STAGE 9: TRAIN ===")
best_val_prauc, best_state, history = -1.0, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); running = 0.0; nb = 0
    for seq, lengths, stat, y in train_loader:
        opt.zero_grad()
        logits = model(seq.to(DEVICE), lengths, stat.to(DEVICE))
        loss = crit(logits, y.to(DEVICE))
        loss.backward(); opt.step()
        running += loss.item(); nb += 1
    yv, pv = evaluate_probs(val_loader)
    val_prauc = average_precision_score(yv, pv)
    val_auroc = roc_auc_score(yv, pv)
    history.append({"epoch": epoch, "train_loss": running/nb, "val_pr_auc": val_prauc, "val_auroc": val_auroc})
    flag = ""
    if val_prauc > best_val_prauc:
        best_val_prauc = val_prauc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        flag = "  <- best"
    print(f"[train] epoch {epoch}/{EPOCHS} loss {running/nb:.4f} | val PR-AUC {val_prauc:.4f} "
          f"AUROC {val_auroc:.4f} | {time.time()-t0:.1f}s{flag}")

model.load_state_dict(best_state)
print(f"[train] best val PR-AUC = {best_val_prauc:.4f}")


=== STAGE 9: TRAIN ===
[train] epoch 1/4 loss 2.7070 | val PR-AUC 0.0239 AUROC 0.5110 | 16.6s  <- best
[train] epoch 2/4 loss 2.5065 | val PR-AUC 0.0242 AUROC 0.5130 | 23.3s  <- best
[train] epoch 3/4 loss 2.4848 | val PR-AUC 0.0237 AUROC 0.5016 | 19.8s
[train] epoch 4/4 loss 2.4696 | val PR-AUC 0.0238 AUROC 0.5002 | 18.6s
[train] best val PR-AUC = 0.0242


## 10 · Evaluate on the held-out TEST split & benchmark vs the rule engine
Primary metric **PR-AUC**; also AUROC, recall at a fixed precision floor, and best-F1.
`rule_engine_baseline_predictions.csv` / `model_verdicts_sample.json` are used **only** here as
external benchmarks — never as inputs.

In [11]:
print("=== STAGE 10: TEST EVALUATION ===")
yt, pt = evaluate_probs(test_loader)
test_prauc = average_precision_score(yt, pt)
test_auroc = roc_auc_score(yt, pt)
prec, rec, thr = precision_recall_curve(yt, pt)
# recall at the first threshold reaching TARGET_PRECISION
ok = np.where(prec[:-1] >= TARGET_PRECISION)[0]
rec_at_p = float(rec[ok].max()) if len(ok) else 0.0
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_f1 = float(np.nanmax(f1s))
print(f"[test] windows {len(yt):,} | fraud {int(yt.sum()):,} ({yt.mean():.4%})")
print(f"[test] PR-AUC ........ {test_prauc:.4f}   (primary)")
print(f"[test] AUROC ......... {test_auroc:.4f}")
print(f"[test] recall @ P>={TARGET_PRECISION:.0%} .. {rec_at_p:.4f}")
print(f"[test] best F1 ....... {best_f1:.4f}")

# ---- Benchmark: rule engine on the SAME test txns (reference only) ----
bench = {}
try:
    test_txn_ids = set(df["txn_id"].values[win_end[test_idx]])
    rule = pd.read_csv(RULE_BASELINE)
    idc = "txn_id" if "txn_id" in rule.columns else rule.columns[0]
    rule = rule[rule[idc].isin(test_txn_ids)]
    dec_col = next((c for c in rule.columns if "decision" in c.lower() or "flag" in c.lower() or "pred" in c.lower()), None)
    lab_map = dict(zip(df["txn_id"].values[win_end[test_idx]], yt))
    if dec_col is not None:
        yr = rule[idc].map(lab_map).values
        pr_flag = rule[dec_col].astype(str).str.upper().isin(["FLAG", "BLOCK", "1", "TRUE", "FRAUD"]).astype(int).values
        keep = ~pd.isna(yr)
        yr, pr_flag = yr[keep].astype(int), pr_flag[keep]
        if len(yr):
            from sklearn.metrics import precision_score, recall_score
            bench = {"rule_auroc_flag": float(roc_auc_score(yr, pr_flag)) if len(np.unique(yr))>1 else None,
                     "rule_precision": float(precision_score(yr, pr_flag, zero_division=0)),
                     "rule_recall": float(recall_score(yr, pr_flag, zero_division=0)),
                     "n": int(len(yr))}
            print(f"[bench] rule-engine on test txns: {bench}")
except Exception as e:
    print("[bench] rule-engine benchmark skipped:", e)

print("\\n[context] Prior diagnostics: this synthetic dump caps offline models at AUROC ~0.52-0.56; "
      "~90% of fraud is statistically inseparable per-transaction. A PR-AUC well above the 1.83% "
      "base rate on the strongly-flagged minority is the realistic outcome, not the 0.93 proposal target.")


=== STAGE 10: TEST EVALUATION ===
[test] windows 47,159 | fraud 883 (1.8724%)
[test] PR-AUC ........ 0.0293   (primary)
[test] AUROC ......... 0.5276
[test] recall @ P>=20% .. 0.0272
[test] best F1 ....... 0.0547
[bench] rule-engine on test txns: {'rule_auroc_flag': 0.5104406306266004, 'rule_precision': 0.02143606655673812, 'rule_recall': 0.16194790486976218, 'n': 47159}
\n[context] Prior diagnostics: this synthetic dump caps offline models at AUROC ~0.52-0.56; ~90% of fraud is statistically inseparable per-transaction. A PR-AUC well above the 1.83% base rate on the strongly-flagged minority is the realistic outcome, not the 0.93 proposal target.


## 11 · Save the model + fitted scalers/encoders + manifest, then read back to verify
Drive/disk writes have truncated before, so every artifact is re-loaded and checked.

In [12]:
print("=== STAGE 11: SAVE + READBACK ===")
MODEL_PATH    = os.path.join(MODELS_DIR, "lstm_two_branch.pt")
SCALERS_PATH  = os.path.join(MODELS_DIR, "preprocessors.joblib")
MANIFEST_PATH = os.path.join(MODELS_DIR, "manifest.json")
METRICS_PATH  = os.path.join(MODELS_DIR, "metrics.json")

torch.save({"state_dict": model.state_dict(),
            "arch": {"n_seq_feat": X.shape[1], "n_static_feat": STATIC_MATRIX.shape[1],
                     "hidden": LSTM_HIDDEN, "layers": LSTM_LAYERS, "static_emb": STATIC_EMB,
                     "fusion_hidden": FUSION_HIDDEN, "dropout": DROPOUT, "seq_len": SEQ_LEN}},
           MODEL_PATH)
joblib.dump({"seq_scaler": seq_scaler, "static_scaler": stat_scaler, "ohe": ohe,
             "district_freq": freq.to_dict()}, SCALERS_PATH)

manifest = {
    "model": "two_branch_lstm",
    "seq_len_N": SEQ_LEN, "n_constraint": "[20,35]; 30 default (txns/account mean~40, 6.9% reach 50)",
    "seq_features": SEQ_FEATURES,
    "static_features": STATIC_FEATURE_NAMES,
    "leakage_excluded": {"posthoc": LEAKAGE_POSTHOC, "other_systems": LEAKAGE_OTHER_SYS,
                          "customer_snapshot": LEAKAGE_SNAPSHOT, "is_fraud_seed": "excluded",
                          "id_strings": DROP_ID_STRING},
    "split_months": {"train": TRAIN_MONTHS, "val": VAL_MONTHS, "test": TEST_MONTHS},
    "train_neg_per_pos": TRAIN_NEG_PER_POS, "pos_weight": POS_WEIGHT,
    "device_timestamp_fix": "min(first_seen,last_seen) applied upstream during cleaning (~50% reversed)",
}
metrics = {"val_pr_auc_best": best_val_prauc, "test_pr_auc": test_prauc, "test_auroc": test_auroc,
           "test_recall_at_p20": rec_at_p, "test_best_f1": best_f1, "history": history,
           "rule_benchmark": bench, "n_test": int(len(yt))}
with open(MANIFEST_PATH, "w") as f: json.dump(manifest, f, indent=2)
with open(METRICS_PATH, "w") as f: json.dump(metrics, f, indent=2, default=float)

# ---- Readback verification ----
ck = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
m2 = TwoBranchLSTM(ck["arch"]["n_seq_feat"], ck["arch"]["n_static_feat"])
m2.load_state_dict(ck["state_dict"]); m2.eval()
pp = joblib.load(SCALERS_PATH)
with torch.no_grad():
    xb, lb, sb, yb = next(iter(test_loader))
    r1 = torch.sigmoid(model(xb.to(DEVICE), lb, sb.to(DEVICE))).cpu()
    r2 = torch.sigmoid(m2(xb, lb, sb))
same = torch.allclose(r1, r2, atol=1e-5)
print(f"[save] model      -> {MODEL_PATH}  ({os.path.getsize(MODEL_PATH):,} B)")
print(f"[save] preproc    -> {SCALERS_PATH}  ({os.path.getsize(SCALERS_PATH):,} B)")
print(f"[save] manifest   -> {MANIFEST_PATH}")
print(f"[save] metrics    -> {METRICS_PATH}")
print(f"[verify] reloaded model reproduces predictions: {same}")
print(f"[verify] preprocessors reloaded: seq_scaler={type(pp['seq_scaler']).__name__}, "
      f"ohe categories ok={hasattr(pp['ohe'],'categories_')}")
assert same, "Readback mismatch — saved model does not reproduce predictions!"
print("\\n[DONE] Two-branch LSTM trained, evaluated, and saved with integrity verified.")


=== STAGE 11: SAVE + READBACK ===
[save] model      -> C:\Developer\Agentic-Fraud-Detection-System-\backend\models\lstm\lstm_two_branch.pt  (183,365 B)
[save] preproc    -> C:\Developer\Agentic-Fraud-Detection-System-\backend\models\lstm\preprocessors.joblib  (5,069 B)
[save] manifest   -> C:\Developer\Agentic-Fraud-Detection-System-\backend\models\lstm\manifest.json
[save] metrics    -> C:\Developer\Agentic-Fraud-Detection-System-\backend\models\lstm\metrics.json
[verify] reloaded model reproduces predictions: True
[verify] preprocessors reloaded: seq_scaler=StandardScaler, ohe categories ok=True
\n[DONE] Two-branch LSTM trained, evaluated, and saved with integrity verified.


## 12 · Summary / README

**What this notebook builds:** a leakage-controlled, time-split, masked two-branch LSTM for
per-transaction fraud detection on the Nepal digital-banking dataset.

**Sequence length N — the [20, 35] constraint.** Full txns-per-account is mean ~40 (min 18,
max 68) but only 6.9% of accounts reach 50 txns, so `N=30` is the default and already near the
ceiling; larger N would make most sequences padding. Sequences are drawn from each account's
**full** history (all 2M txns as context) so N=30 is meaningfully filled — windows are only
allowed to *end* at the 400k labeled txns.

**Device-timestamp fix.** 50.2% of `device_fingerprints.json` rows have `first_seen > last_seen`.
The robust `true_first_seen = min(first_seen, last_seen)` → `device_age_days ≥ 0` derivation was
applied **upstream** during cleaning; this notebook consumes the already-corrected
`feature_table.csv`, and the audit cell re-reports the ~50% reversed rate for traceability.

**Excluded-for-leakage fields.** Post-hoc: `fraud_type`, `fraud_confidence`. Other systems'
outputs (benchmark only): `rule_baseline_decision`, `rule_triggered`, `rule_confidence`,
`model_verdicts_sample.json`. Late customer-profile snapshot: `risk_tier`, `churn_risk_score`,
`avg_monthly_txn_count`, `avg_monthly_txn_value_npr`, `is_dormant`. Graph: `is_fraud_seed`.
High-cardinality IDs/strings dropped at parse time.

**Time-split boundaries.** Train = months 1–8, Val = 9–10, Test = 11–12, on the ending txn's
calendar month. All scalers/encoders are fit on TRAIN only. Legit windows are undersampled 1:10
in TRAIN only (weighted BCE `pos_weight≈53.5`); val/test keep every window.

**Data ceiling (expected, not a defect).** ~90% of fraud in this synthetic dump is statistically
inseparable per-transaction; every offline model here caps at AUROC ~0.52–0.56. The 0.93 proposal
target is unreachable on this dump standalone — it is met only by the fused multi-agent system
(routing ambiguous cases to OTP, high-precision BLOCK on strong deterministic flags). Judge this
model on PR-AUC and its contribution to the fused system, not standalone AUROC.

**Saved artifacts (`backend/models/lstm/`).** `lstm_two_branch.pt` (weights + arch),
`preprocessors.joblib` (seq scaler, static scaler, one-hot encoder, district freq map),
`manifest.json` (exact feature order + all exclusions), `metrics.json`.